In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import (interactive, IntSlider, FloatSlider, Dropdown,
                        VBox, HBox, Button, Output, Text, Label)
from IPython.display import display

In [2]:
def interactive_viewer(image):
    if image.ndim != 3:
        return

    total_channels = image.shape[0]

    channel_slider = IntSlider(min=0, max=total_channels - 1, value=0)

    vmin_slider = FloatSlider(
        min=float(image.min()), max=float(image.max()),
        value=0.0, description='Min:', step=0.1, readout_format='.1f'
    )
    vmax_slider = FloatSlider(
        min=float(image.min()), max=float(image.max()),
        value=1.0, description='Max:', step=0.1, readout_format='.1f'
    )
    cmap_dropdown = Dropdown(
        options=['gray', 'viridis', 'plasma', 'inferno', 'magma', 'cividis',
                 'hot', 'cool', 'spring', 'summer', 'autumn', 'winter',
                 'bone', 'copper', 'RdYlBu', 'Spectral'],
        value='gray', description='Colormap:'
    )
    zoom_slider = FloatSlider(min=0.1, max=3.0, value=1.0, step=0.1, description='Zoom:')

    prev_button = Button(description="◀", layout={'width': '120px'})
    next_button = Button(description="▶", layout={'width': '120px'})
    channel_label = Label(
        value=f"通道: {channel_slider.value + 1} / {total_channels}",
        layout={'align_items': 'center', 'width': '150px'}
    )

    filename_text = Text(value=f'channel_{channel_slider.value}.png', description='文件名:')
    save_button = Button(description="Save", button_style='success', icon='save')

    plot_output = Output()
    save_output = Output()

    def go_previous(_):
        if channel_slider.value > 0:
            channel_slider.value -= 1

    def go_next(_):
        if channel_slider.value < total_channels - 1:
            channel_slider.value += 1

    def on_channel_change(change):
        new_channel_index = change['new']
        current_slice = image[new_channel_index]

        channel_label.value = f"Channel: {new_channel_index + 1} / {total_channels}"
        filename_text.value = f'channel_{new_channel_index + 1}.png'

        vmin_slider.value = np.percentile(current_slice, 1)
        vmax_slider.value = np.percentile(current_slice, 99)

    def save_view(_=None):
        with save_output:
            save_output.clear_output()

            channel = channel_slider.value
            vmin = vmin_slider.value
            vmax = vmax_slider.value
            cmap = cmap_dropdown.value
            filename = filename_text.value

            fig_save, ax_save = plt.subplots(figsize=(8, 8))
            ax_save.imshow(image[channel], cmap=cmap, vmin=vmin, vmax=vmax)
            ax_save.axis('off')

            try:
                fig_save.savefig(filename, dpi=150, bbox_inches='tight', pad_inches=0.1)
                print(f"Pic saved as: '{filename}'")
            except Exception as e:
                raise RuntimeError(f"error saving file '{filename}': {e}")

            plt.close(fig_save)

    def update_plot(channel, vmin, vmax, cmap, zoom):
        with plot_output:
            plot_output.clear_output(wait=True)
            plt.close('all')

            fig = plt.figure(figsize=(8 * zoom, 8 * zoom))
            ax = fig.add_subplot(1, 1, 1)

            im = ax.imshow(image[channel], cmap=cmap, vmin=vmin, vmax=vmax)
            fig.colorbar(im, ax=ax)
            ax.set_title(f'Channel {channel + 1}\nIntensity Range: {vmin:.2f} to {vmax:.2f}')
            ax.axis('off')
            plt.show()

    prev_button.on_click(go_previous)
    next_button.on_click(go_next)
    save_button.on_click(save_view)
    channel_slider.observe(on_channel_change, names='value')

    channel_controls = HBox([prev_button, channel_label, next_button], layout={'justify_content': 'center'})

    controls = VBox([
        channel_controls,
        HBox([vmin_slider, vmax_slider]),
        HBox([cmap_dropdown, zoom_slider]),
        HBox([filename_text, save_button])
    ])

    interactive_plot = interactive(update_plot,
                                  channel=channel_slider,
                                  vmin=vmin_slider,
                                  vmax=vmax_slider,
                                  cmap=cmap_dropdown,
                                  zoom=zoom_slider)

    on_channel_change({'new': channel_slider.value})
    display(VBox([controls, plot_output, save_output]))

In [ ]:
SPNS_patched_file_path_1 = '../datasets/SPNS/Entropy_64_GP_PATCH_224x224_OVERLAP_0x0_MZ_50.0-500.0_BIN_SIZE_0.1/SPNS/2JY1.npz'
patches = np.load(SPNS_patched_file_path_1)['patches']
# print(f"Data shape: {image.shape}")
interactive_viewer(image=patches)

In [8]:
SPNS_patched_file_path_2 = '../datasets/SPNS/Entropy_64_DAPS_PATCH_224x224_WINDOW_100_INT_THR_0.1_DENS_THR_40_MIN_PKS_10_MZ_50.0-500.0_BIN_SIZE_0.1/SPNS/2JY1.npz'
patches = np.load(SPNS_patched_file_path_2)['patches']
# print(f"Data shape: {image.shape}")
interactive_viewer(image=patches)

In [ ]:
CD_patched_file_path_1 = '../datasets/CD/Entropy_64_GP_PATCH_224x224_OVERLAP_0x0_MZ_65.0-1010.0_BIN_SIZE_0.1/CD/01_16_IIRN_Male_CD_2015.npz'
patches = np.load(CD_patched_file_path_1)['patches']
# print(f"Data shape: {image.shape}")
interactive_viewer(image=patches)

In [ ]:
CD_patched_file_path_2 = '../datasets/CD/Entropy_64_DAPS_PATCH_224x224_WINDOW_100_INT_THR_0.1_DENS_THR_45_MIN_PKS_10_MZ_65.0-1010.0_BIN_SIZE_0.1/CD/01_16_IIRN_Male_CD_2015.npz'
patches = np.load(CD_patched_file_path_2)['patches']
# print(f"Data shape: {image.shape}")
interactive_viewer(image=patches)